# Master Pipeline — Penggabungan Dataset YOLOv11
**Project:** Jivara (CC26-PSU090) | **Role:** Data Science  
**Tujuan:** Menggabungkan Dataset Roboflow (29 Kelas) dan Dataset Makanan Indonesia (35 Kelas) menjadi satu Dataset YOLO Terpadu (Unified YOLO Dataset).

Notebook ini membuktikan proses unifikasi kelas semantik, translasi format bounding box, dan auto-zipping yang siap dikirim ke AI Engineer (Roboflow Web).

---
## 1. Initialization & Setup Direktori

In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import yaml
from pathlib import Path

# Mendefinisikan Path Utama
BASE_DIR = Path(r'd:\Dicoding Academy\DataCapstone')
ROBOFLOW_DIR = BASE_DIR / 'data_mentah' / 'Data Gambar Makanan Indonesia'
MI_DIR = BASE_DIR / 'data_mentah' / 'makanan_indonesia'
MI_IMAGE_DIR = MI_DIR / 'food-tfk-images'

OUTPUT_DIR = BASE_DIR / 'data_output' / 'yolo_merged_dataset'

# Reset dan buat direktori output baru
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ['train', 'val', 'test']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print("Setup direktori merger selesai.")

Setup direktori merger selesai.


---
## 2. Unifikasi Kelas Semantik (Semantic Class Merging)

Kedua dataset memiliki kelas yang berbeda, namun ada beberapa kelas yang memiliki makna semantik yang sama. Kita akan menggabungkannya untuk memperkuat performa model.

In [2]:
# 1. Ambil daftar kelas dari Roboflow
robo_classes = sorted(['Apel', 'Ayam Goreng', 'Bakso', 'Biskuit Choco Chips', 'Burger', 'Capcay', 
                'Donat', 'Ikan Goreng', 'Kentang Goreng', 'Kiwi', 'Mie Goreng', 'Nanas', 
                'Nasi Goreng', 'Nasi Putih', 'Nugget', 'Pempek', 'Pisang', 'Pizza', 
                'Rendang Sapi', 'Sate', 'Spaghetti', 'Steak', 'Stroberi', 'Tahu Goreng', 
                'Telur Goreng', 'Telur Rebus', 'Tempe Goreng', 'Terong Balado', 'Tumis Kangkung'])

# 2. Ambil daftar kelas dari Makanan Indonesia
mi_classes = sorted(['asinan-jakarta', 'ayam-betutu', 'ayam-bumbu-rujak', 'ayam-goreng-lengkuas', 
              'bika-ambon', 'bir-pletok', 'bubur-manado', 'cendol', 'es-dawet', 'gado-gado', 
              'gudeg', 'gulai-ikan-mas', 'keladi', 'kerak-telor', 'klappertart', 'kolak', 
              'kue-lumpur', 'kunyit-asam', 'laksa-bogor', 'lumpia-semarang', 'mie-aceh', 
              'nagasari', 'nasi-goreng-kampung', 'papeda', 'pempek-palembang', 'rawon-surabaya', 
              'rendang', 'rujak-cingur', 'sate-ayam-madura', 'sate-lilit', 'sate-maranggi', 
              'soerabi', 'soto-ayam-lamongan', 'soto-banjar', 'tahu-telur'])

# 3. Buat Master Dictionary Mapping
# Kita arahkan kelas semantik mirip menjadi satu nama kelas baru
CLASS_MAPPING_RULES = {
    'Rendang Sapi': 'Rendang',
    'rendang': 'Rendang',
    'Pempek': 'Pempek',
    'pempek-palembang': 'Pempek',
    'Nasi Goreng': 'Nasi Goreng',
    'nasi-goreng-kampung': 'Nasi Goreng',
    'Sate': 'Sate Umum' # Sate lilit, madura, maranggi tetap terpisah
}

unified_classes = set()

# Terapkan aturan mapping
def get_unified_name(original_name):
    return CLASS_MAPPING_RULES.get(original_name, original_name)

for cls in robo_classes + mi_classes:
    unified_classes.add(get_unified_name(cls))

UNIFIED_CLASS_LIST = sorted(list(unified_classes))
CLASS_TO_ID = {name: idx for idx, name in enumerate(UNIFIED_CLASS_LIST)}

print(f"Total kelas Roboflow          : {len(robo_classes)}")
print(f"Total kelas Makanan Indonesia : {len(mi_classes)}")
print(f"Total kelas final (Unified)   : {len(UNIFIED_CLASS_LIST)}\n")

print("Contoh Mapping:")
print(f"  Rendang Sapi     -> {get_unified_name('Rendang Sapi')} (ID: {CLASS_TO_ID[get_unified_name('Rendang Sapi')]})")
print(f"  rendang          -> {get_unified_name('rendang')} (ID: {CLASS_TO_ID[get_unified_name('rendang')]})")
print(f"  sate-ayam-madura -> {get_unified_name('sate-ayam-madura')} (ID: {CLASS_TO_ID[get_unified_name('sate-ayam-madura')]})")

Total kelas Roboflow          : 29
Total kelas Makanan Indonesia : 35
Total kelas final (Unified)   : 61

Contoh Mapping:
  Rendang Sapi     -> Rendang (ID: 18)
  rendang          -> Rendang (ID: 18)
  sate-ayam-madura -> sate-ayam-madura (ID: 54)


---
## 3. Eksekusi Dataset 1 (Roboflow)

Membersihkan invalid bbox dan mengonversi ke format YOLO menggunakan *Unified Class ID*.

In [3]:
def process_roboflow_dataset(raw_dir, output_dir, class_mapping, get_unified_name_func):
    """Proses dataset Roboflow dan copy ke direktori unified."""
    splits = {'train': 'train', 'valid': 'val', 'test': 'test'}
    total_images = 0
    
    for split_src, split_dst in splits.items():
        csv_path = raw_dir / split_src / '_annotations.csv'
        if not csv_path.exists(): continue
            
        df = pd.read_csv(csv_path)
        # Filter invalid bbox
        df = df[~((df['xmin'] >= df['xmax']) | (df['ymin'] >= df['ymax']))]
        
        for filename, group in df.groupby('filename'):
            src_img = raw_dir / split_src / filename
            dst_img = output_dir / 'images' / split_dst / filename
            dst_lbl = output_dir / 'labels' / split_dst / (Path(filename).stem + '.txt')
            
            if not src_img.exists(): continue
                
            shutil.copy2(src_img, dst_img)
            total_images += 1
            
            label_lines = []
            for _, row in group.iterrows():
                unified_name = get_unified_name_func(row['class'])
                class_id = class_mapping[unified_name]
                
                # Normalisasi Bbox
                w, h = row['width'], row['height']
                xc = ((row['xmin'] + row['xmax']) / 2) / w
                yc = ((row['ymin'] + row['ymax']) / 2) / h
                bw = (row['xmax'] - row['xmin']) / w
                bh = (row['ymax'] - row['ymin']) / h
                
                label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
                
            with open(dst_lbl, 'w') as f:
                f.write('\n'.join(label_lines))
                
    return total_images

count_robo = process_roboflow_dataset(ROBOFLOW_DIR, OUTPUT_DIR, CLASS_TO_ID, get_unified_name)
print(f"Berhasil memproses {count_robo} gambar Roboflow.")

Berhasil memproses 3720 gambar Roboflow.


---
## 4. Eksekusi Dataset 2 (Makanan Indonesia)

Dataset ini tidak punya bounding box, jadi kita membuat *synthetic bounding box* (90% ukuran frame) dan mengonversi one-hot encoding.

In [4]:
def process_mi_dataset(raw_dir, img_dir, output_dir, class_mapping, get_unified_name_func):
    """Proses dataset Makanan Indonesia dengan sintesis bbox."""
    splits = {'train.csv': 'train', 'dev.csv': 'val', 'test.csv': 'test'}
    total_images = 0
    
    # Koordinat Synthetic Bbox
    XC, YC, BW, BH = 0.5, 0.5, 0.9, 0.9
    
    for csv_file, split_dst in splits.items():
        csv_path = raw_dir / csv_file
        if not csv_path.exists(): continue
            
        df = pd.read_csv(csv_path)
        class_columns = df.columns[3:]
        df['active_class'] = df[class_columns].idxmax(axis=1)
        
        for _, row in df.iterrows():
            filename = row['Image Index']
            src_img = img_dir / filename
            
            # Gunakan prefix 'MI_' agar nama file tidak bentrok dengan Roboflow
            new_filename = f"MI_{filename}"
            dst_img = output_dir / 'images' / split_dst / new_filename
            dst_lbl = output_dir / 'labels' / split_dst / (Path(new_filename).stem + '.txt')
            
            if not src_img.exists(): continue
                
            shutil.copy2(src_img, dst_img)
            total_images += 1
            
            unified_name = get_unified_name_func(row['active_class'])
            class_id = class_mapping[unified_name]
            
            label_line = f"{class_id} {XC:.6f} {YC:.6f} {BW:.6f} {BH:.6f}\n"
            with open(dst_lbl, 'w') as f:
                f.write(label_line)
                
    return total_images

count_mi = process_mi_dataset(MI_DIR, MI_IMAGE_DIR, OUTPUT_DIR, CLASS_TO_ID, get_unified_name)
print(f"Berhasil memproses {count_mi} gambar Makanan Indonesia.")

Berhasil memproses 1643 gambar Makanan Indonesia.


---
## 5. Pembuatan `data.yaml` dan Kompresi (Zipping)

Membuat konfigurasi standar YOLO dan langsung men-ZIP foldernya agar siap dikirim.

In [5]:
# 1. Generate data.yaml
data_config = {
    'path': str(OUTPUT_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(UNIFIED_CLASS_LIST),
    'names': UNIFIED_CLASS_LIST
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml terbuat dengan {data_config['nc']} kelas.")

# 2. Kompresi Folder ke ZIP
import zipfile

zip_path = BASE_DIR / 'data_output' / 'merged_dataset_final.zip'

def zipdir(path, ziph):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            archive_path = os.path.relpath(file_path, os.path.join(path, '..'))
            ziph.write(file_path, archive_path)

print(f"\nSedang melakukan kompresi ke {zip_path}... (Mohon tunggu beberapa saat)")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(OUTPUT_DIR, zipf)

print("✅ Selesai! File ZIP berhasil dibuat.")
print("File ini siap di-drag & drop ke Roboflow oleh teman Anda.")

data.yaml terbuat dengan 61 kelas.

Sedang melakukan kompresi ke d:\Dicoding Academy\DataCapstone\data_output\merged_dataset_final.zip... (Mohon tunggu beberapa saat)
✅ Selesai! File ZIP berhasil dibuat.
File ini siap di-drag & drop ke Roboflow oleh teman Anda.
